# Backblaze Hard Drive Failure Prediction Pipeline

End-to-end pipeline covering:
1. **Data Cleaning** — column filtering and missing value imputation
2. **Temporal Aggregation** — rolling window features per disk
3. **RFOD Anomaly Scores** — in-notebook RFOD training and multi-threshold scoring
4. **XGBoost Classification** — RUL class prediction with hyperparameter tuning
5. **Cost Matrix Evaluation** — asymmetric cost-based RUL assessment

In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score,
    log_loss, precision_recall_fscore_support,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import (
    GroupKFold, RandomizedSearchCV, StratifiedGroupKFold,
    cross_val_predict,
)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import randint, uniform
from xgboost import XGBClassifier

PROJECT_DIR = Path(".").resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from backblaze_feature_engineering import (
    clean_train_val_datasets,
    build_feature_dataset,
    temporal_aggregation,
    generate_validation_labels,
    patch_missing_values,
)
from train_xgboost_backblaze import (
    COST_MATRIX,
    total_cost,
    align_feature_frames,
    build_model_pipeline,
    tune_pipeline,
    evaluate_predictions,
    load_aggregated_train_val,
)

RANDOM_STATE = 42
print("Imports OK")

In [ ]:
DATA_DIR      = PROJECT_DIR / "Datasets" / "Backblaze" / "backblaze_data"
CLEAN_DIR     = PROJECT_DIR / "Datasets" / "Backblaze" / "clean"

TRAIN_RAW     = DATA_DIR / "train_set.csv"
VAL_RAW       = DATA_DIR / "val_set.csv"
VAL_IDS       = DATA_DIR / "val_serial_number_id.csv"
VAL_LABELS    = DATA_DIR / "val_label.csv"

TRAIN_AGG_OUT = CLEAN_DIR / "backblaze_clean_train_feature_agg.csv"
VAL_AGG_OUT   = CLEAN_DIR / "backblaze_clean_val_feature_agg.csv"

WINDOW_SIZE   = 10
SAMPLE_FRAC   = 0.5   # fraction of normal samples used for RFOD training

print(f"Train raw : {TRAIN_RAW}  exists={TRAIN_RAW.exists()}")
print(f"Val raw   : {VAL_RAW}    exists={VAL_RAW.exists()}")

---
## 1. Data Cleaning

- Drop columns with >80 % missing values
- Drop constant and duplicate columns
- Keep raw SMART metrics, drop redundant normalized counterparts
- Impute remaining NaNs fitted on train only

In [ ]:
print("Loading raw CSVs...")
train_raw = pd.read_csv(TRAIN_RAW, parse_dates=["date"], low_memory=False)
val_raw   = pd.read_csv(VAL_RAW,   parse_dates=["date"], low_memory=False)
print(f"Train raw shape: {train_raw.shape}")
print(f"Val raw shape  : {val_raw.shape}")
print(f"Failure rate in train: {train_raw['failure'].mean():.4%}")

In [ ]:
train_clean, val_clean, report = clean_train_val_datasets(train_raw, val_raw)

print("\n── Cleaning report ──")
print(f"  Kept columns        : {len(report.kept_columns)}")
print(f"  Dropped high-missing: {len(report.dropped_high_missing)}")
print(f"  Dropped constant    : {len(report.dropped_constant)}")
print(f"  Dropped duplicate   : {len(report.dropped_duplicate)}")
print(f"  Dropped redundant   : {len(report.dropped_redundant_smart)} (normalized, kept raw)")
print(f"\nTrain clean shape: {train_clean.shape}")
print(f"Val clean shape  : {val_clean.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
miss_before = train_raw.isna().mean().sort_values(ascending=False)
miss_after  = train_clean.isna().mean().sort_values(ascending=False)
axes[0].bar(range(len(miss_before)), miss_before.values)
axes[0].set_title("Missing ratio per column — BEFORE cleaning")
axes[0].set_xlabel("Column index"); axes[0].set_ylabel("Missing ratio")
axes[1].bar(range(len(miss_after)), miss_after.values, color="steelblue")
axes[1].set_title("Missing ratio per column — AFTER cleaning")
axes[1].set_xlabel("Column index"); axes[1].set_ylabel("Missing ratio")
plt.tight_layout(); plt.show()

---
## 2. Temporal Aggregation

Rolling window aggregation (mean) over `WINDOW_SIZE` days per disk.  
Also computes RUL and 3-class labels:
- **0** — RUL ≥ 20 days (healthy)
- **1** — 10 ≤ RUL < 20 days (warning)
- **2** — RUL < 10 days (critical)

In [ ]:
print(f"Building training feature dataset (window={WINDOW_SIZE})...")
train_agg = build_feature_dataset(
    train_clean,
    window_size=WINDOW_SIZE,
    censored_rul_value=9999,
    drop_censored=False,
    patch_missing=False,
)
print(f"Train agg shape: {train_agg.shape}")
print("Label distribution:")
print(train_agg["label"].value_counts().sort_index())

In [ ]:
print(f"Building validation feature dataset (window={WINDOW_SIZE})...")
val_ids_df    = pd.read_csv(VAL_IDS)
val_labels_df = pd.read_csv(VAL_LABELS)

val_agg_features = temporal_aggregation(
    val_clean,
    window_size=WINDOW_SIZE,
)
val_agg = generate_validation_labels(
    val_agg_features, val_ids_df, val_labels_df,
    labeling_mode="final_only",
)
print(f"Val agg shape: {val_agg.shape}")
print("Val label distribution:")
print(val_agg["label"].value_counts().sort_index())

---
## 3. RFOD Anomaly Scores

RFOD (Random Forest Outlier Detection) trains one random forest per feature to predict
that feature from all others. Anomaly score = weighted Gower distance between
prediction and true value across all features.

Multiple thresholds `k` are used: for each `k`, RFOD is retrained treating
labels `0..k` as normal and `> k` as anomaly, producing `anomaly_score_k` columns.

### 3.1 RFOD Helper Functions

In [ ]:
# Alpha values for Gower distance IQR normalisation
RFOD_ALPHA = [0, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.25]


def smape(y_true, y_pred):
    return np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-5))


def alphaquantile(matrix1, matrix2, categorical_columns, continuous_columns,
                  column_names, model_scores, residual, weighted=True):
    """Compute improved Gower distance for multiple alpha thresholds."""
    n_rows, n_cols = matrix1.shape
    result = {}
    for a in RFOD_ALPHA:
        diff_matrix = np.zeros((n_rows, n_cols))
        for col in continuous_columns:
            col_name = column_names[col]
            iqr = np.percentile(residual[col_name], 100 * (1 - a)) * 2
            if iqr == 0:
                scaled_diff = np.zeros_like(matrix1[:, col])
            else:
                scaled_diff = np.clip(np.abs(matrix1[:, col] - matrix2[:, col]) / iqr, 0, 1)
            diff_matrix[:, col] = scaled_diff
        for col_idx in categorical_columns:
            diff_matrix[:, col_idx] = 1 - matrix1[:, col_idx]  # false proba
        if weighted:
            mean_diff = np.sum(diff_matrix * np.array(model_scores), axis=1)
        else:
            mean_diff = np.mean(diff_matrix, axis=1)
        result[a] = (diff_matrix, mean_diff)
    return result


def calc_resi(col_name, Xy, treeid, model):
    X_resi = Xy.drop(columns=[col_name])
    y_resi = Xy[col_name]
    preds = sum(model.estimators_[idx].predict(X_resi.values) for idx in treeid)
    return np.abs(preds / len(treeid) - y_resi.values)


print("RFOD helper functions defined.")

In [ ]:
# These two functions reference the global `_rfod_col_name` set during the training loop
_rfod_col_name = None  # will be set in the training loop


def _tree_scoring(tree, X, y, col_name, num_cols):
    if col_name in num_cols:
        y_pred = tree.predict(X)
        return smape(y, y_pred), y_pred
    else:
        y_pred = tree.predict_proba(X)
        n_cls = y_pred.shape[1]
        if n_cls != len(np.unique(y)) or n_cls <= 1:
            score = 0
        elif n_cls == 2:
            score = roc_auc_score(y, y_pred[:, 1])
        else:
            score = roc_auc_score(y, y_pred, multi_class="ovr")
        return 1 - score, y_pred


def _trees_scoring(y_preds, y, col_name, num_cols):
    y_preds = np.mean(y_preds, axis=0)
    if col_name in num_cols:
        return smape(y, y_preds)
    else:
        n_cls = y_preds.shape[1]
        if n_cls != len(np.unique(y)) or n_cls <= 1:
            return 1.0
        elif n_cls == 2:
            return 1 - roc_auc_score(y, y_preds[:, 1])
        else:
            return 1 - roc_auc_score(y, y_preds, multi_class="ovr")


def train_rf_feature(col_name, num_cols, categ_cols,
                     X_processed_train, X_processed_valid,
                     X_train, X_valid, process_list):
    """Train one RF per feature and select best tree subset via golden-section search."""
    rf_X_train = X_processed_train.drop(columns=process_list[col_name])
    rf_X_valid = X_processed_valid.drop(columns=process_list[col_name])
    rf_y_train = X_train[col_name]
    rf_y_valid = X_valid[col_name]

    if col_name in num_cols:
        rf_model = RandomForestRegressor(n_estimators=400, n_jobs=-1)
    else:
        rf_model = RandomForestClassifier(n_estimators=400, n_jobs=-1)

    t0 = time.time()
    rf_model.fit(rf_X_train.values, rf_y_train)
    tr_time = time.time() - t0

    trees = rf_model.estimators_
    t0 = time.time()
    output_trees = Parallel(n_jobs=-1)(
        delayed(_tree_scoring)(tree, rf_X_valid.values, rf_y_valid.values, col_name, num_cols)
        for tree in trees
    )
    errors, preds = zip(*output_trees)
    tree_oob_errors = np.asarray(errors, float)
    y_preds = np.stack(preds, axis=0)
    tree_indices = np.argsort(tree_oob_errors)
    sorted_errors = tree_oob_errors[tree_indices]

    # Golden-section search for best tree count
    trees_oob_errors = {}
    lef, rig = 0, 399
    while lef + 3 <= rig:
        midl = int(rig - (rig - lef) * 0.61803398875)
        midr = int(lef + (rig - lef) * 0.61803398875)
        for mid in (midl, midr):
            if mid not in trees_oob_errors:
                trees_oob_errors[mid] = (
                    _trees_scoring(y_preds[tree_indices[:mid + 1]], rf_y_valid.values, col_name, num_cols)
                    + sorted_errors[0] * 0.3 / np.sqrt((mid + 1) / 10)
                )
        if trees_oob_errors[midl] < trees_oob_errors[midr]:
            rig = midr
        else:
            lef = midl

    Best_trees = 5
    trees_oob_errors.setdefault(
        Best_trees,
        _trees_scoring(y_preds[tree_indices[:Best_trees + 1]], rf_y_valid.values, col_name, num_cols)
        + sorted_errors[0] * 0.3 / np.sqrt((Best_trees + 1) / 10)
    )
    for i in range(lef, rig + 1):
        trees_oob_errors.setdefault(
            i,
            _trees_scoring(y_preds[tree_indices[:i + 1]], rf_y_valid.values, col_name, num_cols)
            + sorted_errors[0] * 0.3 / np.sqrt((i + 1) / 10)
        )
        if trees_oob_errors[i] < trees_oob_errors[Best_trees]:
            Best_trees = i

    pr_time = time.time() - t0
    return {
        "tree_indices": tree_indices,
        "rf_model": rf_model,
        "tr_time": tr_time,
        "pr_time": pr_time,
        "tree_used": max(Best_trees, 5),
        "feature_performance": float(np.mean(sorted_errors[:max(1, Best_trees - 1)])),
    }


def predict_feature(col_name, tree_used, true_value, tree_indices, rf_model, num_cols,
                    X_processed_test, process_list):
    """Predict one feature from all others; return confidence scores and predictions."""
    top_idx = tree_indices[:tree_used]
    trees   = rf_model.estimators_
    predict_X = X_processed_test.drop(columns=process_list[col_name]).to_numpy()

    if col_name in num_cols:
        top_preds = np.array([trees[i].predict(predict_X) for i in top_idx])
        final_pred = top_preds.mean(axis=0)
        confidence_scores = np.std(top_preds, axis=0, ddof=1)
    else:
        class_idx_map = {label: i for i, label in enumerate(rf_model.classes_)}
        map_func = np.vectorize(lambda x: class_idx_map.get(x, -1))
        true_cls = map_func(true_value)
        valid = true_cls != -1
        true_proba = np.zeros((predict_X.shape[0], len(top_idx)))
        for j, idx in enumerate(top_idx):
            tp = trees[idx].predict_proba(predict_X)
            tt = np.zeros(predict_X.shape[0])
            tt[valid] = tp[np.arange(predict_X.shape[0])[valid], true_cls[valid]]
            true_proba[:, j] = tt
        final_pred = true_proba.mean(axis=1)
        confidence_scores = np.std(true_proba, axis=1, ddof=1)

    return {"confidence_scores": confidence_scores, "final_pred": final_pred}


print("RFOD core functions defined.")

### 3.2 Data Preparation for RFOD

In [ ]:
_meta_cols = {"date", "serial_number", "failure", "failure_date", "rul_days", "label"}

# --- train: failed disks (rul_days <= 60) ---
_bb_failed = train_agg[train_agg["rul_days"] <= 60].reset_index(drop=True)
_feat_cols = [c for c in _bb_failed.columns if c not in _meta_cols]

# --- train: sample normal rows from never-failed disks (rul_days == 9999) ---
CENSORED_SAMPLE_FRAC = 0.05   # adjust as needed
_bb_censored = (
    train_agg[train_agg["rul_days"] == 9999]
    .sample(frac=CENSORED_SAMPLE_FRAC, random_state=42)
    .reset_index(drop=True)
)
# label these as 0 (normal)
_bb_censored = _bb_censored.copy()
_bb_censored["label"] = 0

_bb_train = pd.concat([_bb_failed, _bb_censored], ignore_index=True)
X_bb_train = _bb_train[_feat_cols].reset_index(drop=True)
y_bb_train = _bb_train["label"].reset_index(drop=True)

print(f"Failed disk rows  : {len(_bb_failed)}")
print(f"Censored disk rows: {len(_bb_censored)}  (frac={CENSORED_SAMPLE_FRAC})")
print(f"Total train rows  : {len(_bb_train)}")
print(f"Label dist in train: {y_bb_train.value_counts().sort_index().to_dict()}")

# --- val: last window per disk (label already attached in Section 2) ---
_bb_val = val_agg.copy()
_bb_val["date"] = pd.to_datetime(_bb_val["date"], errors="coerce")
_bb_val_last = (
    _bb_val.sort_values(["serial_number", "date"])
           .drop_duplicates("serial_number", keep="last")
)
_val_feat_cols = [c for c in _feat_cols if c in _bb_val_last.columns]
X_bb_val = _bb_val_last[_val_feat_cols].reset_index(drop=True)
y_bb_val = _bb_val_last["label"].astype(int).reset_index(drop=True)

# Combined X / y_raw for RFOD (train + val)
X_all = pd.concat([X_bb_train[_val_feat_cols], X_bb_val], ignore_index=True)
y_raw = pd.concat([y_bb_train, y_bb_val], ignore_index=True)

# Drop all-NaN columns
always_nan = [c for c in X_all.columns if X_all[c].isna().all()]
X_all = X_all.drop(columns=always_nan)

y_binary = (y_raw > 0).astype(int)  # binary anomaly label for RFOD training

categ_cols = [c for c in ["model"] if c in X_all.columns]
num_cols   = [c for c in X_all.columns if c not in categ_cols]
fitonly    = []

print(f"X_all shape : {X_all.shape}")
print(f"y_raw dist  : {y_raw.value_counts().sort_index().to_dict()}")
print(f"y_binary    : anomaly={y_binary.sum()}, normal={( y_binary==0).sum()}")
print(f"Categorical : {categ_cols}")
print(f"Numerical   : {len(num_cols)} columns")

### 3.3 Preprocessing (LabelEncoder + OHE + StandardScaler)

In [ ]:
X_all = X_all.replace("?", np.nan)

# LabelEncode categoricals, median-fill numericals
_label_encoders = {}
_num_medians    = {}
for col in X_all.columns:
    if col in categ_cols:
        X_all[col] = X_all[col].fillna(X_all[col].mode()[0])
        le = LabelEncoder()
        X_all[col] = le.fit_transform(X_all[col])
        _label_encoders[col] = le
    else:
        _num_medians[col] = X_all[col].median()
        X_all[col] = X_all[col].fillna(_num_medians[col])

y_binary = (y_raw > 0).astype(int)

print(f"Label dist: {y_raw.value_counts().sort_index().to_dict()}")
print(f"normal={( y_binary==0).sum()}, anomaly={(y_binary==1).sum()}")

# Train / valid / test split（只用真实样本，不做 SMOTE）
X_train_rfod = X_all[y_binary == 0].sample(frac=0.6, random_state=42)
X_valid_rfod = X_train_rfod.sample(frac=0.1 / 0.6, random_state=42)
X_test_rfod  = pd.concat([X_all[y_binary == 0].drop(X_train_rfod.index),
                           X_all[y_binary == 1]])
y_test_rfod  = y_binary.loc[X_test_rfod.index]

# Apply sample_frac
X_train_rfod = X_train_rfod.sample(frac=SAMPLE_FRAC, random_state=42)
X_valid_rfod = X_valid_rfod.sample(frac=SAMPLE_FRAC, random_state=42)

for df in (X_train_rfod, X_valid_rfod, X_test_rfod):
    df.reset_index(drop=True, inplace=True)
y_test_rfod = y_test_rfod.reset_index(drop=True)

print(f"X_train_rfod: {X_train_rfod.shape}")
print(f"X_valid_rfod: {X_valid_rfod.shape}")
print(f"X_test_rfod : {X_test_rfod.shape}  anomaly rate: {y_test_rfod.mean():.3%}")


In [ ]:
# OHE for categoricals, StandardScaler for numericals
process_list     = {}
_onehot_encoders = {}
Xp_train = pd.DataFrame()
Xp_valid = pd.DataFrame()
Xp_test  = pd.DataFrame()

for col in categ_cols:
    enc = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
    ohe_tr = enc.fit_transform(X_train_rfod[[col]])
    _onehot_encoders[col] = enc
    process_list[col] = [f"{col}_{cat}" for cat in enc.categories_[0][1:]]
    Xp_train = pd.concat([Xp_train, pd.DataFrame(ohe_tr,                           columns=process_list[col])], axis=1)
    Xp_valid = pd.concat([Xp_valid, pd.DataFrame(enc.transform(X_valid_rfod[[col]]), columns=process_list[col])], axis=1)
    Xp_test  = pd.concat([Xp_test,  pd.DataFrame(enc.transform(X_test_rfod[[col]]),  columns=process_list[col])], axis=1)

for col in num_cols:
    process_list[col] = [col]

scaler = StandardScaler()
Xp_train = pd.concat([Xp_train, pd.DataFrame(scaler.fit_transform(X_train_rfod[num_cols]), columns=num_cols)], axis=1)
Xp_valid = pd.concat([Xp_valid, pd.DataFrame(scaler.transform(X_valid_rfod[num_cols]),     columns=num_cols)], axis=1)
Xp_test  = pd.concat([Xp_test,  pd.DataFrame(scaler.transform(X_test_rfod[num_cols]),      columns=num_cols)], axis=1)

for df in (Xp_train, Xp_valid, Xp_test):
    df.reset_index(drop=True, inplace=True)

print(f"Xp_train: {Xp_train.shape}")
print(f"Xp_test : {Xp_test.shape}")

### 3.4 Train Per-Feature Random Forests

In [ ]:
Tid          = {}
rFM          = {}
Feature_tree_used = {}

t_start = time.time()
for col in X_train_rfod.columns:
    if col in fitonly:
        continue
    print(f"  Training RF for: {col}")
    res = train_rf_feature(
        col, num_cols, categ_cols,
        Xp_train, Xp_valid,
        X_train_rfod, X_valid_rfod,
        process_list,
    )
    Tid[col]               = res["tree_indices"]
    rFM[col]               = res["rf_model"]
    Feature_tree_used[col] = res["tree_used"]

print(f"\nTotal training time: {time.time() - t_start:.1f}s")

### 3.5 Compute Residuals and Score Test Set

In [ ]:
# Residuals on validation set (used for IQR normalisation in alphaquantile)
residual = {}
for col in num_cols:
    residual[col] = np.sort(calc_resi(
        col, Xp_valid,
        Tid[col][:Feature_tree_used[col]], rFM[col],
    ))

# Predict each feature on test set
Conf_Pre = {}
for col in X_train_rfod.columns:
    if col in fitonly:
        continue
    Conf_Pre[col] = predict_feature(
        col, Feature_tree_used[col],
        X_test_rfod[col].values,
        Tid[col], rFM[col], num_cols,
        Xp_test, process_list,
    )

predcol = [c for c in X_train_rfod.columns if c not in fitonly]
conf_combined = pd.DataFrame({c: Conf_Pre[c]["confidence_scores"] for c in predcol})[predcol]
all_y_pred_rf = pd.DataFrame({c: Conf_Pre[c]["final_pred"]        for c in predcol})[predcol]

# Feature weights (normalised inverse confidence)
fw = conf_combined.to_numpy().copy()
for i in range(fw.shape[0]):
    s = fw[i].sum()
    if s: fw[i] /= s
fw = 1 - fw
for i in range(fw.shape[0]):
    s = fw[i].sum()
    if s: fw[i] /= s
fw = np.square(fw)
fw = fw / np.sum(fw, axis=1, keepdims=True)

X_test_pred = X_test_rfod.drop(columns=fitonly)
categ_idx = [i for i, c in enumerate(X_test_pred.columns) if c in categ_cols]
num_idx   = [i for i, c in enumerate(X_test_pred.columns) if c in num_cols]

result = alphaquantile(
    all_y_pred_rf.values, X_test_pred.values,
    categ_idx, num_idx, X_test_pred.columns,
    fw, residual, weighted=True,
)

print("Alphaquantile scoring done.")

### 3.6 Alpha Sweep → Best AUCROC Alpha

In [ ]:
best_aucroc       = -1
best_aucroc_alpha = None   # AUCROC-optimal alpha (for display only)
best_avpr         = -1
best_avpr_alpha   = None   # AUPR-optimal alpha   (used for scoring)
alpha_metrics     = []

thresholds = np.arange(0.001, 1.001, 0.001)

for a, (_, scores_a) in result.items():
    aucroc = roc_auc_score(y_test_rfod, scores_a)
    avpr   = average_precision_score(y_test_rfod, scores_a)

    best_f1_a = max(
        f1_score(y_test_rfod, (scores_a > t).astype(int), zero_division=0)
        for t in thresholds
    )
    alpha_metrics.append({"alpha": a, "aucroc": aucroc, "avpr": avpr, "best_f1": best_f1_a})

    if aucroc > best_aucroc:
        best_aucroc       = aucroc
        best_aucroc_alpha = a
    if avpr > best_avpr:
        best_avpr         = avpr
        best_avpr_alpha   = a

# Use AUPR (more robust under class imbalance) for final alpha selection
best_alpha_base = best_avpr_alpha

alpha_df = pd.DataFrame(alpha_metrics).set_index("alpha")
print(f"Best AUCROC: {best_aucroc:.4f}  (alpha={best_aucroc_alpha})")
print(f"Best AUPR  : {best_avpr:.4f}  (alpha={best_avpr_alpha})  \u2190 selected as best_alpha_base")
print()
print(alpha_df.round(4).to_string())

### 3.7 Multi-Threshold Retraining and Scoring

For each label threshold `k ∈ {1, 2}`, retrain RFOD treating labels `0..k` as normal.
Then score the full train and val CSVs with `anomaly_score_0`, `anomaly_score_1`, `anomaly_score_2`.

In [ ]:
def _safe_le_transform(le, values):
    arr = np.asarray(values)
    known = np.array([v in set(le.classes_) for v in arr])
    result = np.full(len(arr), -1, dtype=np.int64)
    if known.any():
        result[known] = le.transform(arr[known])
    return result


_agg_feat_cols = list(X_all.columns)


def _preprocess_for_scoring(df_orig, sc=None, ohe_k=None, plist_k=None):
    """Preprocess a DataFrame using base (or threshold-k) encoders."""
    _ohe   = ohe_k   if ohe_k   is not None else _onehot_encoders
    _plist = plist_k if plist_k is not None else process_list
    _sc    = sc      if sc      is not None else scaler

    df = df_orig[[c for c in _agg_feat_cols if c in df_orig.columns]].copy().replace("?", np.nan)
    available_cols = df.columns.tolist()

    for col in available_cols:
        if col in categ_cols:
            fill = df[col].mode()
            df[col] = df[col].fillna(fill[0] if len(fill) > 0 else 0)
            df[col] = _safe_le_transform(_label_encoders[col], df[col].values)
        else:
            df[col] = df[col].fillna(_num_medians.get(col, 0))
    df.reset_index(drop=True, inplace=True)

    X_proc = pd.DataFrame()
    for col in categ_cols:
        if col not in _ohe:
            continue
        ohe_vals = _ohe[col].transform(df[[col]])
        X_proc = pd.concat([X_proc, pd.DataFrame(ohe_vals, columns=_plist[col])], axis=1)
    nc = [c for c in num_cols if c in available_cols]
    X_proc = pd.concat([X_proc, pd.DataFrame(_sc.transform(df[nc]), columns=nc)], axis=1)
    X_proc.reset_index(drop=True, inplace=True)
    return df, X_proc


def _score_rows(X_raw, X_proc, alpha_val, Ftree, rfm, Tid_, resid, pl):
    """Compute RFOD anomaly scores for a batch of rows."""
    Conf = {}
    for col in X_raw.columns:
        if col in fitonly or col not in Ftree:
            continue
        Conf[col] = predict_feature(
            col, Ftree[col], X_raw[col].values,
            Tid_[col], rfm[col], num_cols, X_proc, pl,
        )

    pcols = [c for c in X_raw.columns if c not in fitonly and c in Ftree]
    conf_ = pd.DataFrame({c: Conf[c]["confidence_scores"] for c in pcols})[pcols]
    pred_ = pd.DataFrame({c: Conf[c]["final_pred"]        for c in pcols})[pcols]

    fw_ = conf_.to_numpy().copy()
    for i in range(fw_.shape[0]):
        s = fw_[i].sum()
        if s: fw_[i] /= s
    fw_ = 1 - fw_
    for i in range(fw_.shape[0]):
        s = fw_[i].sum()
        if s: fw_[i] /= s
    fw_ = np.square(fw_)
    fw_ = fw_ / np.sum(fw_, axis=1, keepdims=True)

    ci = [i for i, c in enumerate(X_raw.columns) if c in categ_cols]
    ni = [i for i, c in enumerate(X_raw.columns) if c in num_cols]
    res_ = alphaquantile(pred_.values, X_raw.values, ci, ni, X_raw.columns, fw_, resid, weighted=True)
    _, scores = res_[alpha_val]
    return scores


def _find_best_alpha(X_te, Xp_te, y_te, Ftree, rfm, Tid_, resid, pl):
    """Alpha sweep on a test set; return alpha with best AUPR."""
    Conf = {}
    for col in X_te.columns:
        if col in fitonly or col not in Ftree:
            continue
        Conf[col] = predict_feature(
            col, Ftree[col], X_te[col].values,
            Tid_[col], rfm[col], num_cols, Xp_te, pl,
        )

    pcols = [c for c in X_te.columns if c not in fitonly and c in Ftree]
    conf_ = pd.DataFrame({c: Conf[c]["confidence_scores"] for c in pcols})[pcols]
    pred_ = pd.DataFrame({c: Conf[c]["final_pred"]        for c in pcols})[pcols]

    fw_ = conf_.to_numpy().copy()
    for i in range(fw_.shape[0]):
        s = fw_[i].sum()
        if s: fw_[i] /= s
    fw_ = 1 - fw_
    for i in range(fw_.shape[0]):
        s = fw_[i].sum()
        if s: fw_[i] /= s
    fw_ = np.square(fw_)
    fw_ = fw_ / np.sum(fw_, axis=1, keepdims=True)

    ci = [i for i, c in enumerate(X_te.columns) if c in categ_cols]
    ni = [i for i, c in enumerate(X_te.columns) if c in num_cols]
    result = alphaquantile(pred_.values, X_te.values, ci, ni, X_te.columns, fw_, resid, weighted=True)

    best_alpha, best_aupr = None, -1
    for a, (_, scores_a) in result.items():
        if y_te.nunique() < 2:
            continue
        aupr = average_precision_score(y_te, scores_a)
        if aupr > best_aupr:
            best_aupr  = aupr
            best_alpha = a
    return best_alpha, best_aupr


def _retrain_for_threshold(k):
    """Retrain RFOD treating y_raw <= k as normal; find best alpha for this k."""
    y_k = (y_raw.values > k).astype(int)
    mask_normal = y_k == 0

    _Xtr = X_all[mask_normal].sample(frac=0.6, random_state=42)
    _Xvl = _Xtr.sample(frac=0.1 / 0.6, random_state=42)
    _Xte_normal  = X_all[mask_normal].drop(_Xtr.index)
    _Xte_anomaly = X_all[~mask_normal]
    _Xte = pd.concat([_Xte_normal, _Xte_anomaly])
    _yte = pd.Series(
        [0] * len(_Xte_normal) + [1] * len(_Xte_anomaly),
        name="y",
    ).reset_index(drop=True)

    _Xtr = _Xtr.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)
    _Xvl = _Xvl.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)
    _Xte = _Xte.reset_index(drop=True)

    _plist_k, _ohe_k = {}, {}
    _Xp_tr = pd.DataFrame(); _Xp_vl = pd.DataFrame(); _Xp_te = pd.DataFrame()
    for c in categ_cols:
        enc = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
        tr_ohe = enc.fit_transform(_Xtr[[c]])
        _ohe_k[c]   = enc
        _plist_k[c] = [f"{c}_{cat}" for cat in enc.categories_[0][1:]]
        _Xp_tr = pd.concat([_Xp_tr, pd.DataFrame(tr_ohe,                    columns=_plist_k[c])], axis=1)
        _Xp_vl = pd.concat([_Xp_vl, pd.DataFrame(enc.transform(_Xvl[[c]]), columns=_plist_k[c])], axis=1)
        _Xp_te = pd.concat([_Xp_te, pd.DataFrame(enc.transform(_Xte[[c]]), columns=_plist_k[c])], axis=1)

    for c in num_cols:
        _plist_k[c] = [c]

    _sc_k = StandardScaler()
    nc_k  = [c for c in num_cols if c in _Xtr.columns]
    _Xp_tr = pd.concat([_Xp_tr, pd.DataFrame(_sc_k.fit_transform(_Xtr[nc_k]), columns=nc_k)], axis=1)
    _Xp_vl = pd.concat([_Xp_vl, pd.DataFrame(_sc_k.transform(_Xvl[nc_k]),    columns=nc_k)], axis=1)
    _Xp_te = pd.concat([_Xp_te, pd.DataFrame(_sc_k.transform(_Xte[nc_k]),    columns=nc_k)], axis=1)
    for df in (_Xp_tr, _Xp_vl, _Xp_te):
        df.reset_index(drop=True, inplace=True)

    _Ftree_k, _rFM_k, _Tid_k = {}, {}, {}
    for col in _Xtr.columns:
        if col in fitonly:
            continue
        r = train_rf_feature(col, num_cols, categ_cols, _Xp_tr, _Xp_vl, _Xtr, _Xvl, _plist_k)
        _Tid_k[col]   = r["tree_indices"]
        _rFM_k[col]   = r["rf_model"]
        _Ftree_k[col] = r["tree_used"]

    _resid_k = {
        col: np.sort(calc_resi(col, _Xp_vl, _Tid_k[col][:_Ftree_k[col]], _rFM_k[col]))
        for col in num_cols if col not in fitonly
    }

    # Find best alpha for this threshold k (AUPR-based)
    _best_alpha_k, _best_aupr_k = _find_best_alpha(
        _Xte, _Xp_te, _yte, _Ftree_k, _rFM_k, _Tid_k, _resid_k, _plist_k
    )

    # Additional metrics at best alpha
    _scores_best = _score_rows(_Xte, _Xp_te, _best_alpha_k,
                                _Ftree_k, _rFM_k, _Tid_k, _resid_k, _plist_k)
    _thresholds = np.arange(0.001, 1.001, 0.001)
    _best_f1, _best_thr = max(
        (f1_score(_yte, (_scores_best > t).astype(int), zero_division=0), t)
        for t in _thresholds
    )
    _y_bin = (_scores_best > _best_thr).astype(int)
    _prec  = precision_score(_yte, _y_bin, zero_division=0)
    _rec   = recall_score(_yte, _y_bin, zero_division=0)
    _aucroc_k = roc_auc_score(_yte, _scores_best) if _yte.nunique() >= 2 else float("nan")
    print(
        f"  k={k}: best_alpha={_best_alpha_k}"
        f"  AUPR={_best_aupr_k:.4f}  AUCROC={_aucroc_k:.4f}"
        f"  F1={_best_f1:.4f}(thr={_best_thr:.3f})"
        f"  Prec={_prec:.4f}  Rec={_rec:.4f}"
        f"  pos_ratio={_yte.mean():.3f}"
    )

    return _Ftree_k, _rFM_k, _Tid_k, _resid_k, _sc_k, _ohe_k, _plist_k, _best_alpha_k


print("Scoring helpers defined.")

In [ ]:
# Determine label thresholds (exclude max label — no anomalies left)
_all_labels     = sorted(int(v) for v in y_raw.unique() if pd.notna(v))
_score_thresholds = _all_labels[:-1]
print(f"Label values: {_all_labels}  →  score thresholds: {_score_thresholds}")

extra_models = {}
for k in _score_thresholds:
    if k == 0:
        continue
    print(f"\nRetraining RFOD for threshold k={k} (normal = labels 0..{k})...")
    extra_models[k] = _retrain_for_threshold(k)  # returns (..., best_alpha_k)
    print(f"Done k={k}")

### 3.8 Score Train and Validation Set


In [ ]:
print("Scoring train set (all at once)...")

train_scored = train_agg.copy()
xr0, xp0 = _preprocess_for_scoring(train_scored)
train_scored["anomaly_score_0"] = _score_rows(
    xr0, xp0, best_alpha_base, Feature_tree_used, rFM, Tid, residual, process_list
)
for k, arts in extra_models.items():
    Ft_k, rfm_k, Tid_k, resid_k, sc_k, ohe_k, plist_k, alpha_k = arts
    xrk, xpk = _preprocess_for_scoring(train_scored, sc=sc_k, ohe_k=ohe_k, plist_k=plist_k)
    train_scored[f"anomaly_score_{k}"] = _score_rows(
        xrk, xpk, alpha_k, Ft_k, rfm_k, Tid_k, resid_k, plist_k
    )

print(f"Train scored: {len(train_scored)} rows (in memory)")


In [ ]:
print("Scoring validation set...")

# Reconstruct val with label column (last readout per disk)
_bb_val_full = val_agg.copy()
_bb_val_full["date"] = pd.to_datetime(_bb_val_full["date"], errors="coerce")
_lbl_map = (
    _bb_val_full.sort_values(["serial_number", "date"])
    .drop_duplicates("serial_number", keep="last")[["serial_number", "label"]]
    .rename(columns={"label": "_lbl"})
)
_bb_val_full = _bb_val_full.drop(columns=["label"], errors="ignore")
_bb_val_full = _bb_val_full.merge(_lbl_map, on="serial_number", how="left")
_bb_val_full["label"] = _bb_val_full["_lbl"].fillna(-1).astype(int)
_bb_val_full.drop(columns=["_lbl"], inplace=True)
_bb_val_full.reset_index(drop=True, inplace=True)

val_scored = _bb_val_full.copy()

vr0, vp0 = _preprocess_for_scoring(_bb_val_full)
val_scored["anomaly_score_0"] = _score_rows(vr0, vp0, best_alpha_base,
                                              Feature_tree_used, rFM, Tid, residual, process_list)
for k, arts in extra_models.items():
    Ft_k, rfm_k, Tid_k, resid_k, sc_k, ohe_k, plist_k, alpha_k = arts
    vrk, vpk = _preprocess_for_scoring(_bb_val_full, sc=sc_k, ohe_k=ohe_k, plist_k=plist_k)
    val_scored[f"anomaly_score_{k}"] = _score_rows(vrk, vpk, alpha_k,
                                                    Ft_k, rfm_k, Tid_k, resid_k, plist_k)

print(f"Val scored: {len(val_scored)} rows (in memory)")


In [ ]:
score_cols = [c for c in train_scored.columns if c.startswith("anomaly_score")]
print(f"Anomaly score columns: {score_cols}")
print(f"Train scored shape: {train_scored.shape}")
print(f"Val scored shape  : {val_scored.shape}")
train_scored[score_cols].describe().round(4)


In [ ]:
if score_cols and "label" in train_scored.columns:
    n = len(score_cols)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 4), squeeze=False)
    for ax, sc in zip(axes[0], score_cols):
        for lbl, grp in train_scored.groupby("label"):
            ax.hist(grp[sc].dropna(), bins=50, alpha=0.5, label=f"label={lbl}")
        ax.set_title(sc); ax.set_xlabel("Anomaly score"); ax.legend()
    plt.suptitle("Anomaly score distributions by RUL label (train)")
    plt.tight_layout(); plt.show()

print("Spearman correlation (anomaly score vs label):")
for sc in score_cols:
    corr = train_scored[[sc, "label"]].dropna().corr(method="spearman").loc[sc, "label"]
    print(f"  {sc}: {corr:.4f}")

In [ ]:
# ── XGBoost 数据准备：保留全量训练数据，用 sample_weight 处理类别不均衡 ─────────
# XGBoost 是有监督方法，正常样本越多越好；用逆类频率 sample_weight 代替采样/SMOTE
_drop_for_xgb  = {"date", "serial_number", "failure", "failure_date", "rul_days", "label"}
_xgb_feat_cols = [c for c in train_scored.columns if c not in _drop_for_xgb]

X_train_xgb_raw   = train_scored[_xgb_feat_cols].reset_index(drop=True)
y_train_xgb_raw   = train_scored["label"].astype(int).reset_index(drop=True)
groups_raw        = train_scored["serial_number"].reset_index(drop=True)
sample_weight_raw = compute_sample_weight("balanced", y=y_train_xgb_raw)

print(f"Full train label dist: {y_train_xgb_raw.value_counts().sort_index().to_dict()}")
print(f"sample_weight  label-0={sample_weight_raw[y_train_xgb_raw==0].mean():.4f}  "
      f"label-1={sample_weight_raw[y_train_xgb_raw==1].mean():.4f}  "
      f"label-2={sample_weight_raw[y_train_xgb_raw==2].mean():.4f}")


---
## 4. XGBoost Classification

Train an XGBoost multi-class classifier (3 classes: label 0/1/2) using
the temporally aggregated features + RFOD anomaly scores.
Hyperparameters are tuned via `RandomizedSearchCV` with `StratifiedGroupKFold`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

COST_MATRIX = np.array(
    [[0, 1, 3],
     [4, 0, 2],
     [15, 5, 0]],
    dtype=int,
)

def total_cost(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    return int(COST_MATRIX[y_true, y_pred].sum())


def align_feature_frames(X_train, X_val):
    all_nan = [c for c in X_train.columns if X_train[c].isna().all()]
    if all_nan:
        X_train = X_train.drop(columns=all_nan)
        X_val   = X_val.drop(columns=all_nan, errors="ignore")
    X_val = X_val.reindex(columns=X_train.columns)
    return X_train, X_val


def build_model_pipeline(X_train):
    cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
    num_cols = [c for c in X_train.columns if c not in cat_cols]

    preprocessor = ColumnTransformer(transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot",  OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ])
    model = XGBClassifier(
        objective="multi:softprob", num_class=3, eval_metric="mlogloss",
        tree_method="hist", device="cuda",  # GPU 加速，无 GPU 时改为 device="cpu"
        random_state=RANDOM_STATE, n_jobs=1,
    )
    return Pipeline([("preprocessor", preprocessor), ("model", model)])


def build_cv(y_train, groups, n_splits):
    try:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
        list(cv.split(np.zeros(len(y_train)), y_train, groups))
        return cv
    except Exception:
        return GroupKFold(n_splits=n_splits)


def tune_pipeline(pipeline, X_train, y_train, groups, n_splits, n_iter,
                  sample_weight=None):
    cv = build_cv(y_train, groups, n_splits)
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions={
            "model__n_estimators":     randint(150, 450),
            "model__max_depth":        randint(3, 9),
            "model__learning_rate":    uniform(0.03, 0.17),
            "model__subsample":        uniform(0.65, 0.35),
            "model__colsample_bytree": uniform(0.65, 0.35),
            "model__min_child_weight": randint(1, 8),
            "model__gamma":            uniform(0.0, 2.0),
            "model__reg_lambda":       uniform(0.5, 2.5),
            "model__reg_alpha":        uniform(0.0, 1.0),
        },
        n_iter=n_iter, scoring="f1_macro", cv=cv,
        refit=True, random_state=RANDOM_STATE, n_jobs=1, verbose=1,
    )
    fit_kw = {"groups": groups}
    if sample_weight is not None:
        fit_kw["model__sample_weight"] = sample_weight
    search.fit(X_train, y_train, **fit_kw)
    return search


def evaluate_predictions(y_true, y_pred, y_pred_proba, label=""):
    aucroc_ovr = roc_auc_score(y_true, y_pred_proba, multi_class="ovr", average="macro")
    aucroc_ovo = roc_auc_score(y_true, y_pred_proba, multi_class="ovo", average="macro")
    return {
        "label":             label,
        "accuracy":          accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1":          f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1":       f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "aucroc_ovr_macro":  aucroc_ovr,
        "aucroc_ovo_macro":  aucroc_ovo,
        "total_cost":        total_cost(y_true, y_pred),
    }


# ── 从 Cell 33 准备的全量数据切分特征/标签 ───────────────────────────────────
_drop_val = ["date", "serial_number", "failure", "label"]

# 验证集：每块磁盘只取最后一条记录
_val_last = (
    val_scored
    .sort_values(["serial_number", "date"], kind="mergesort")
    .drop_duplicates("serial_number", keep="last")
    .reset_index(drop=True)
)

X_train_xgb   = X_train_xgb_raw
y_train_xgb   = y_train_xgb_raw
groups        = groups_raw
sample_weight = sample_weight_raw

X_val_xgb = _val_last.drop(columns=[c for c in _drop_val if c in _val_last.columns])
y_val_xgb = _val_last["label"].astype(int).reset_index(drop=True)

X_train_xgb, X_val_xgb = align_feature_frames(X_train_xgb.copy(), X_val_xgb.copy())

print(f"X_train: {X_train_xgb.shape}  label dist: {y_train_xgb.value_counts().sort_index().to_dict()}")
print(f"X_val  : {X_val_xgb.shape}    label dist: {y_val_xgb.value_counts().sort_index().to_dict()}")


In [ ]:
N_SPLITS = 3
N_ITER   = 20

pipeline = build_model_pipeline(X_train_xgb)
search = tune_pipeline(
    pipeline=pipeline,
    X_train=X_train_xgb, y_train=y_train_xgb, groups=groups,
    n_splits=N_SPLITS, n_iter=N_ITER,
    sample_weight=sample_weight,
)

best_model   = search.best_estimator_
y_pred       = best_model.predict(X_val_xgb)
y_pred_proba = best_model.predict_proba(X_val_xgb)

m = evaluate_predictions(y_val_xgb, y_pred, y_pred_proba, label="argmax")
print(f"Best CV macro-F1 : {search.best_score_:.4f}")
print()
print("── argmax validation metrics ──")
for k in ("accuracy","balanced_accuracy","macro_f1","weighted_f1","aucroc_ovr_macro","total_cost"):
    print(f"  {k:25s}: {m[k]:.4f}" if isinstance(m[k], float) else f"  {k:25s}: {m[k]}")
print()
print(classification_report(y_val_xgb, y_pred, labels=[0,1,2],
      target_names=["0 (healthy)","1 (warning)","2 (critical)"], zero_division=0))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_val_xgb, y_pred, labels=[0,1,2])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred 0","Pred 1","Pred 2"],
            yticklabels=["True 0","True 1","True 2"], ax=axes[0])
axes[0].set_title("Confusion Matrix — argmax")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

# Feature importance
xgb_step = best_model.named_steps["model"]
pre_step  = best_model.named_steps["preprocessor"]
try:
    feat_names = pre_step.get_feature_names_out()
except Exception:
    feat_names = [f"f{i}" for i in range(len(xgb_step.feature_importances_))]
importances = pd.Series(xgb_step.feature_importances_, index=feat_names).nlargest(20)
importances.sort_values().plot.barh(ax=axes[1])
axes[1].set_title("Top 20 Feature Importances (XGBoost)")
axes[1].set_xlabel("Importance")

plt.tight_layout(); plt.show()


---
## 5. Cost Matrix RUL Prediction

Asymmetric cost matrix (rows = true label, cols = predicted label):
```
       Pred 0  Pred 1  Pred 2
True 0    0       1       3
True 1    4       0       2
True 2   15       5       0
```
Compare default `argmax` against expected-cost-minimising predictions.

In [ ]:
def threshold_predict(y_pred_proba, theta_1, theta_2):
    """
    Ordered 3-class thresholding:
      predict 2 if P(Critical) >= theta_2
      else predict 1 if P(Warning) + P(Critical) >= theta_1
      else predict 0
    """
    p_wc = y_pred_proba[:, 1] + y_pred_proba[:, 2]
    p_c  = y_pred_proba[:, 2]
    y_pred = np.zeros(len(y_pred_proba), dtype=int)
    y_pred[p_wc >= theta_1] = 1
    y_pred[p_c  >= theta_2] = 2
    return y_pred


def search_optimal_thresholds(
    y_true, y_pred_proba,
    threshold_steps=20,
    theta_1_min=0.2, theta_1_max=0.8,
    theta_2_min=0.1, theta_2_max=0.8,
    safe_recall_min=0.7,
    enforce_theta_order=True,
    fallback_to_unconstrained=True,
):
    """Grid-search (theta_1, theta_2) to minimise total cost."""
    y_true  = np.asarray(y_true, dtype=int)
    t1_vals = np.linspace(theta_1_min, theta_1_max, threshold_steps)
    t2_vals = np.linspace(theta_2_min, theta_2_max, threshold_steps)
    p_wc    = y_pred_proba[:, 1] + y_pred_proba[:, 2]
    p_c     = y_pred_proba[:, 2]
    safe_mask  = y_true == 0
    safe_count = int(safe_mask.sum())

    def _search(t1s, t2s, constrained):
        best_cost, best_t1, best_t2, best_pred = None, 0.5, 0.5, None
        for t1 in t1s:
            for t2 in t2s:
                if constrained and enforce_theta_order and t2 > t1:
                    continue
                yp = np.zeros(len(y_true), dtype=int)
                yp[p_wc >= t1] = 1
                yp[p_c  >= t2] = 2
                if constrained and safe_count > 0:
                    if float((yp[safe_mask] == 0).sum()) / safe_count < safe_recall_min:
                        continue
                c = int(COST_MATRIX[y_true, yp].sum())
                if best_cost is None or c < best_cost:
                    best_cost, best_t1, best_t2, best_pred = c, float(t1), float(t2), yp.copy()
        return best_cost, best_t1, best_t2, best_pred

    cost, t1, t2, pred = _search(t1_vals, t2_vals, constrained=True)
    used_constraints = True

    if cost is None and fallback_to_unconstrained:
        used_constraints = False
        print("No feasible constrained pair found; falling back to unconstrained search.")
        fb = np.linspace(0.0, 1.0, threshold_steps)
        cost, t1, t2, pred = _search(fb, fb, constrained=False)

    return {"theta_1": t1, "theta_2": t2, "total_cost": cost,
            "y_pred": pred, "used_constraints": used_constraints}


print("Threshold functions defined.")


In [ ]:
print("Cost matrix:")
print(pd.DataFrame(COST_MATRIX,
    index=["True 0","True 1","True 2"],
    columns=["Pred 0","Pred 1","Pred 2"]).to_string())

# ── Strategy 1: argmax (already computed) ────────────────────────────────────
m_argmax = evaluate_predictions(y_val_xgb, y_pred, y_pred_proba, label="argmax")

# ── Strategy 2: threshold moving (grid search on theta_1, theta_2) ───────────
thr_result = search_optimal_thresholds(
    y_val_xgb, y_pred_proba,
    threshold_steps=30,
    theta_1_min=0.2, theta_1_max=0.8,
    theta_2_min=0.1, theta_2_max=0.8,
    safe_recall_min=0.7,
    enforce_theta_order=True,
)
y_pred_thr = thr_result["y_pred"]
m_thr = evaluate_predictions(y_val_xgb, y_pred_thr, y_pred_proba, label="threshold_moving")
print(f"\nThreshold moving: theta_1={thr_result['theta_1']:.3f}  "
      f"theta_2={thr_result['theta_2']:.3f}  "
      f"constraints_used={thr_result['used_constraints']}")

# ── Strategy 3: expected cost minimisation (fixed matrix) ────────────────────
y_pred_ecm = np.argmin(y_pred_proba @ COST_MATRIX, axis=1)
m_ecm = evaluate_predictions(y_val_xgb, y_pred_ecm, y_pred_proba, label="expected_cost_min")

# ── Strategy 4: cost matrix perturbation（delta 在训练集 OOF 上搜索，避免验证集泄漏）──
# 用 cross_val_predict 在训练集上得到无偏 OOF 概率，再搜索最优扰动 delta
print("Computing OOF predictions for delta search (refits model per fold)...")
_cv_oof = build_cv(y_train_xgb, groups, N_SPLITS)
y_oof_proba = _cvp(
    best_model, X_train_xgb, y_train_xgb,
    cv=_cv_oof, method="predict_proba",
    groups=groups.values,
    params={"model__sample_weight": sample_weight},
)

_offdiag_idx = [(r, c_) for r in range(3) for c_ in range(3) if r != c_]
_delta_range  = np.linspace(-5, 5, 41)   # 在训练 OOF 上粗搜

best_oof_cost, best_delta = None, 0.0
for _d in _delta_range:
    _cm_try = COST_MATRIX.astype(float).copy()
    for _r, _c in _offdiag_idx:
        _cm_try[_r, _c] = max(0.0, _cm_try[_r, _c] + _d)
    _yp_oof = np.argmin(y_oof_proba @ _cm_try, axis=1)
    _c_oof  = int(COST_MATRIX[y_train_xgb.values.astype(int), _yp_oof].sum())
    if best_oof_cost is None or _c_oof < best_oof_cost:
        best_oof_cost = _c_oof
        best_delta    = float(_d)

# 用最优 delta 在验证集上评估（最终结果无数据泄漏）
best_cm_perturb = COST_MATRIX.astype(float).copy()
for _r, _c in _offdiag_idx:
    best_cm_perturb[_r, _c] = max(0.0, best_cm_perturb[_r, _c] + best_delta)
y_pred_perturb    = np.argmin(y_pred_proba @ best_cm_perturb, axis=1)
best_cost_perturb = int(COST_MATRIX[np.asarray(y_val_xgb, int), y_pred_perturb].sum())

m_perturb = evaluate_predictions(y_val_xgb, y_pred_perturb, y_pred_proba, label="cost_matrix_perturb")
print(f"\nBest delta={best_delta:+.1f}  OOF cost={best_oof_cost}  val cost={best_cost_perturb}")
print("Perturbed cost matrix:")
print(pd.DataFrame(best_cm_perturb.round(1),
      index=["True 0","True 1","True 2"],
      columns=["Pred 0","Pred 1","Pred 2"]).to_string())

# ── Side-by-side summary ─────────────────────────────────────────────────────
summary_cols = ["accuracy","balanced_accuracy","macro_f1","total_cost"]
summary = pd.DataFrame([m_argmax, m_thr, m_ecm, m_perturb]).set_index("label")[summary_cols]
print("\n── Strategy comparison ──")
print(summary.round(4).to_string())


In [ ]:
strategies = [
    ("argmax",                y_pred),
    ("threshold_moving",      y_pred_thr),
    ("expected_cost_min",     y_pred_ecm),
    ("cost_matrix_perturb",   y_pred_perturb),
]
tick_labels = ["0 (healthy)","1 (warning)","2 (critical)"]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, (name, preds) in zip(axes.flat, strategies):
    cm = confusion_matrix(y_val_xgb, preds, labels=[0,1,2])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=tick_labels, yticklabels=tick_labels, ax=ax)
    ax.set_title(f"{name}\ncost={total_cost(y_val_xgb, preds)}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.suptitle("Prediction strategy comparison — Validation Set")
plt.tight_layout(); plt.show()


In [ ]:
y_true_arr = np.asarray(y_val_xgb, dtype=int)
breakdown = pd.DataFrame({
    "true_label":          y_true_arr,
    "cost_argmax":         COST_MATRIX[y_true_arr, np.asarray(y_pred,          dtype=int)],
    "cost_threshold":      COST_MATRIX[y_true_arr, np.asarray(y_pred_thr,      dtype=int)],
    "cost_expected_cost":  COST_MATRIX[y_true_arr, np.asarray(y_pred_ecm,      dtype=int)],
    "cost_cm_perturb":     COST_MATRIX[y_true_arr, np.asarray(y_pred_perturb,  dtype=int)],
})
print("Mean cost per true label class:")
print(breakdown.groupby("true_label")[
    ["cost_argmax","cost_threshold","cost_expected_cost","cost_cm_perturb"]
].mean().round(3).to_string())


In [ ]:
# ── Per-class classification performance for each strategy ───────────────────
_strategies_preds = [
    ("argmax",              y_pred),
    ("threshold_moving",    y_pred_thr),
    ("expected_cost_min",   y_pred_ecm),
    ("cost_matrix_perturb", y_pred_perturb),
]
_class_names = ["0-healthy", "1-warning", "2-critical"]
_y_true = np.asarray(y_val_xgb, dtype=int)

rows = []
for strat_name, preds in _strategies_preds:
    preds = np.asarray(preds, dtype=int)
    p, r, f, sup = precision_recall_fscore_support(
        _y_true, preds, labels=[0, 1, 2], zero_division=0
    )
    for i, cls in enumerate(_class_names):
        rows.append({
            "strategy":  strat_name,
            "class":     cls,
            "precision": p[i],
            "recall":    r[i],
            "f1":        f[i],
            "support":   int(sup[i]),
        })

per_class_df = pd.DataFrame(rows)
pivot_f1   = per_class_df.pivot(index="strategy", columns="class", values="f1").round(3)
pivot_prec = per_class_df.pivot(index="strategy", columns="class", values="precision").round(3)
pivot_rec  = per_class_df.pivot(index="strategy", columns="class", values="recall").round(3)

print("── F1 per class ──")
print(pivot_f1.to_string())
print()
print("── Precision per class ──")
print(pivot_prec.to_string())
print()
print("── Recall per class ──")
print(pivot_rec.to_string())

# ── Bar chart: F1 per class per strategy ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
for ax, cls in zip(axes, _class_names):
    sub = per_class_df[per_class_df["class"] == cls]
    bars = ax.bar(sub["strategy"], sub["f1"], color=["#4C72B0","#DD8452","#55A868","#C44E52"])
    ax.set_title(f"F1 — {cls}")
    ax.set_ylim(0, 1)
    ax.set_ylabel("F1")
    ax.tick_params(axis="x", rotation=20)
    for bar, val in zip(bars, sub["f1"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
plt.suptitle("Per-class F1 by strategy — Validation Set")
plt.tight_layout()
plt.show()

# ── Recall bar chart (most important for safety-critical classes) ─────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
for ax, cls in zip(axes, _class_names):
    sub = per_class_df[per_class_df["class"] == cls]
    bars = ax.bar(sub["strategy"], sub["recall"], color=["#4C72B0","#DD8452","#55A868","#C44E52"])
    ax.set_title(f"Recall — {cls}")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Recall")
    ax.tick_params(axis="x", rotation=20)
    for bar, val in zip(bars, sub["recall"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
plt.suptitle("Per-class Recall by strategy — Validation Set")
plt.tight_layout()
plt.show()
